# Laboratorio 3 — Detección de Anomalías con Machine Learning
Autor: Jorge Luis Gutierrez Miranda — UPEU, Ingeniería de Sistemas, ciclo IX

Dataset: `lab3/network_traffic.csv` (10 000 registros de tráfico de red, 30 días)

## Tarea 3.1 — Exploración y preprocesamiento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import joblib

sns.set_theme()

df = pd.read_csv('lab3/network_traffic.csv', parse_dates=['timestamp'])
print(df.shape)
df.head()

In [ ]:
df.describe(include='all')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['bytes_sent'], bins=50, ax=axes[0])
axes[0].set_title('Distribución de bytes_sent')
sns.histplot(df['duration_sec'], bins=50, ax=axes[1])
axes[1].set_title('Distribución de duration_sec')
plt.tight_layout()
plt.show()

In [ ]:
# Nulos
print(df.isnull().sum())
df = df.dropna(subset=['bytes_sent', 'bytes_recv', 'duration_sec', 'packets'])

# Atípicos extremos: recorte por percentil 99.5 en variables de volumen
for col in ['bytes_sent', 'bytes_recv', 'duration_sec', 'packets']:
    limite = df[col].quantile(0.995)
    df = df[df[col] <= limite]

print('Registros tras limpieza:', len(df))

In [ ]:
# Feature engineering
df['ratio_bytes'] = df['bytes_sent'] / df['bytes_recv'].replace(0, 1)
df['bytes_por_segundo'] = (df['bytes_sent'] + df['bytes_recv']) / df['duration_sec'].replace(0, 1)

df[['ratio_bytes', 'bytes_por_segundo']].describe()

In [ ]:
features = ['bytes_sent', 'bytes_recv', 'dst_port', 'duration_sec', 'packets', 'ratio_bytes', 'bytes_por_segundo']
X = df[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=features, index=df.index)
X_scaled.head()

## Tarea 3.2 — Entrenamiento del modelo

In [ ]:
modelo = IsolationForest(contamination=0.05, n_estimators=100, random_state=42)
modelo.fit(X_scaled)

predicciones = modelo.predict(X_scaled)  # -1 = anomalia, 1 = normal
df['prediccion'] = predicciones
df['prediccion'].value_counts()

In [ ]:
y_true = (df['label'] == 'anomaly').astype(int)
y_pred = (df['prediccion'] == -1).astype(int)

precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f'Precision: {precision:.3f}')
print(f'Recall:    {recall:.3f}')
print(f'F1-Score:  {f1:.3f}')

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Anomalía'], yticklabels=['Normal', 'Anomalía'])
plt.xlabel('Predicción')
plt.ylabel('Real (label)')
plt.title('Matriz de confusión')
plt.tight_layout()
plt.show()

## Tarea 3.3 — Interpretación y umbral dinámico

In [ ]:
df['score'] = modelo.decision_function(X_scaled)

plt.figure(figsize=(10, 4))
plt.plot(df['score'].sort_values().values)
plt.axhline(0, color='red', linestyle='--', label='umbral por defecto (0)')
plt.title('Score de anomalía (decision_function) ordenado')
plt.xlabel('Registro (ordenado por score)')
plt.ylabel('Score')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
umbrales = np.linspace(df['score'].min(), df['score'].max(), 200)
f1_scores = []
for u in umbrales:
    pred_u = (df['score'] < u).astype(int)  # < umbral => anomalia
    f1_scores.append(f1_score(y_true, pred_u))

umbral_optimo = umbrales[int(np.argmax(f1_scores))]
f1_optimo = max(f1_scores)
print(f'Umbral óptimo: {umbral_optimo:.4f} -> F1: {f1_optimo:.3f}')

plt.figure(figsize=(10, 4))
plt.plot(umbrales, f1_scores)
plt.axvline(umbral_optimo, color='red', linestyle='--', label=f'umbral óptimo = {umbral_optimo:.3f}')
plt.title('Curva Umbral vs F1-Score')
plt.xlabel('Umbral de score')
plt.ylabel('F1-Score')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
top10_anomalas = df.sort_values('score').head(10)
top10_anomalas

**Interpretación (completar tras ejecutar con datos reales):**

Las 10 conexiones con el score más bajo (más anómalas) suelen compartir alguno de estos rasgos: volúmenes de `bytes_sent` muy por encima de la media (posible exfiltración de datos), `duration_sec` extremadamente corto combinado con `packets` alto (posible escaneo de puertos o flood), o `ratio_bytes` muy desbalanceado (mucho más tráfico saliente que entrante, típico de exfiltración o de un host comprometido actuando como C2 beacon). Puertos de destino poco comunes (`dst_port` fuera del rango de servicios estándar) en conexiones de alto volumen refuerzan la hipótesis de actividad maliciosa más que de tráfico legítimo mal etiquetado.

## Tarea 3.4 — Exportación del modelo

In [ ]:
joblib.dump({'modelo': modelo, 'scaler': scaler, 'features': features, 'umbral': umbral_optimo}, 'lab3/modelo_anomalias.pkl')
print('Modelo guardado en lab3/modelo_anomalias.pkl')